# Clinical Note Analysis: Hybrid Pipeline

This notebook demonstrates a robust pipeline for parsing clinical reports using:
1.  **BioBERT/ClinicalBERT**: Local models for Medical Entity Recognition (NER) and **Vector Embeddings**.
2.  **Semantic Search**: Finding relevant patient histories contextually.
3.  **Mistral API**: LLM for reasoning, summarization, and **Conditional UI Logic**.

**Note**: All logic is contained within this notebook for ease of use.

In [1]:
import os
import json
import logging
from typing import Dict, List, Optional, Union
import requests
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import pandas as pd
import dotenv
# Auto-reload modules 
%load_ext autoreload
%autoreload 2
dotenv.load_dotenv()
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Common Configuration
DATASET_DIR = "./synthetic-medical-dataset"

## 1. ClinicalNoteParser Class
Defines the core logic for NER, Embeddings, and LLM parsing.

In [2]:
class ClinicalNoteParser:
    def __init__(self, mistral_api_key: Optional[str] = None):
        """
        Initialize the Clinical Note Parser.
        
        Args:
            mistral_api_key: API Key for Mistral. If None, checks MISTRAL_API_KEY env var.
        """
        self.mistral_key = mistral_api_key or os.environ.get("MISTRAL_API_KEY")
        
        if not self.mistral_key:
            logger.warning("No Mistral API Key provided. LLM capabilities will be disabled.")
            print("TIP: Set `os.environ['MISTRAL_API_KEY'] = 'your_key'` before initialization.")
        else:
            logger.info("Mistral API Key detected.")

        self.ner_pipeline = None
        self.embedding_pipeline = None

    def load_local_models(self, ner_model: str = "d4data/biomedical-ner-all", embedding_model: str = "emilyalsentzer/Bio_ClinicalBERT", device: int = -1):
        """
        Load local BioBERT/ClinicalBERT models for Entity Recognition and Embeddings.
        
        Args:
            ner_model: Hugging Face model for NER.
            embedding_model: Hugging Face model for Feature Extraction.
            device: Device ID (-1 for CPU, 0+ for GPU).
        """
        logger.info(f"Loading local NER model: {ner_model}...")
        try:
            # Check for GPU
            if torch.cuda.is_available() and device < 0:
                device = 0
                logger.info(f"CUDA detected. Using GPU: {torch.cuda.get_device_name(0)}")
            
            # NER Pipeline
            self.ner_pipeline = pipeline(
                "token-classification", 
                model=ner_model, 
                tokenizer=ner_model, 
                aggregation_strategy="simple",
                device=device
            )
            
            # Embedding Pipeline
            logger.info(f"Loading local Embedding model: {embedding_model}...")
            self.embedding_pipeline = pipeline(
                "feature-extraction",
                model=embedding_model,
                tokenizer=embedding_model,
                device=device
            )

            logger.info("Local models loaded successfully.")
        except Exception as e:
            logger.error(f"Error loading local models: {e}")
            raise

    def get_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate vector embeddings for a list of text chunks.
        """
        if not self.embedding_pipeline:
            logger.warning("Embedding pipeline not loaded. Call load_local_models() first.")
            return np.array([])
        
        try:
            # features is a list of lists of lists (batch x seq_len x hidden_dim)
            features = self.embedding_pipeline(texts, truncation=True, max_length=512)
            
            embeddings = []
            for i, seq_features in enumerate(features):
                # seq_features is a List[List[float]] (seq_len x hidden_dim)
                # Convert to numpy array to ensure consistency
                try:
                    seq_arr = np.array(seq_features)
                    if seq_arr.ndim == 2:
                         # Standard case: (SeqLen, HiddenDim)
                         mean_embedding = np.mean(seq_arr, axis=0)
                    elif seq_arr.ndim == 3:
                         # Unexpected nested case: (1, SeqLen, HiddenDim)
                         mean_embedding = np.mean(seq_arr, axis=1).flatten()
                    else:
                         # Fallback for weird shapes
                         logger.warning(f"Unexpected shape for text {i}: {seq_arr.shape}")
                         mean_embedding = np.zeros(768) # Fallback to zeros if dimensions are wrong
                         
                except Exception as array_err:
                    logger.error(f"Array conversion error for text {i}: {array_err}")
                    mean_embedding = np.zeros(768)

                # Ensure it's 1D
                if mean_embedding.ndim > 1:
                    mean_embedding = mean_embedding.flatten()
                    
                embeddings.append(mean_embedding)
                
            return np.array(embeddings)
        except Exception as e:
            logger.error(f"Error generating embeddings: {e}")
            return np.array([])

    def search_similar(self, query: str, corpus_texts: List[str], top_k: int = 3) -> List[Dict]:
        """
        Semantic search: Find chunks in corpus_texts similar to the query.
        """
        if not self.embedding_pipeline:
            logger.warning("Embedding pipeline not loaded.")
            return []

        query_emb = self.get_embeddings([query]) # [1, hidden_dim]
        corpus_emb = self.get_embeddings(corpus_texts) # [N, hidden_dim]
        
        if len(query_emb) == 0 or len(corpus_emb) == 0:
            return []

        # Cosine Similarity
        # Reshape for sklearn if needed, but [1, dim] and [N, dim] works
        similarities = cosine_similarity(query_emb, corpus_emb)[0] # [N] scores
        
        # Get indices of top k
        top_indices = similarities.argsort()[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                "text": corpus_texts[idx],
                "score": float(similarities[idx])
            })
            
        return results

    def extract_entities_bert(self, text: str) -> List[Dict]:
        """
        Extract medical entities using the local BERT model.
        """
        if not self.ner_pipeline:
            logger.warning("NER pipeline not loaded. Call load_local_models() first.")
            return []
        
        try:
            entities = self.ner_pipeline(text)
            cleaned_entities = []
            for entity in entities:
                cleaned_entities.append({
                    "entity_group": entity.get("entity_group"),
                    "score": float(entity.get("score", 0.0)),
                    "word": entity.get("word"),
                    "start": entity.get("start"),
                    "end": entity.get("end")
                })
            return cleaned_entities
        except Exception as e:
            logger.error(f"Error during BERT extraction: {e}")
            return []

    def analyze_ui_hints(self, structured_data: Dict) -> Dict:
        """
        Analyze the parsed data to generate UI hints.
        Determines which charts or components should be triggered.
        """
        hints = {
            "show_bp_trend": False,
            "show_medication_list": False,
            "show_lab_table": False,
            "critical_alert": False
        }
        
        if not structured_data:
            return hints

        # logic for BP Trend
        vitals = structured_data.get("vital_signs", {})
        if isinstance(vitals, list): # If it's a list of readings
             if len(vitals) > 1:
                 hints["show_bp_trend"] = True
        elif isinstance(vitals, dict):
            bp = vitals.get("blood_pressure") or vitals.get("bp")
            if isinstance(bp, list) and len(bp) > 1:
                 hints["show_bp_trend"] = True

        # Logic for Med List
        meds = structured_data.get("medications", [])
        if meds and len(meds) > 0:
            hints["show_medication_list"] = True

        # Logic for Critical Alert
        plan = str(structured_data.get("plan", "")).lower()
        assessment = str(structured_data.get("diagnosis", "")).lower()
        critical_keywords = ["urgent", "emergency", "crisis", "critical", "severe"]
        if any(w in plan for w in critical_keywords) or any(w in assessment for w in critical_keywords):
            hints["critical_alert"] = True
            
        return hints

    def parse_with_mistral(self, text: str, context: Optional[List[Dict]] = None) -> Dict:
        """
        Parse the clinical note into a structured JSON using Mistral API via HTTP requests.
        """
        if not self.mistral_key:
            return {"error": "Mistral API Key missing"}

        context_str = ""
        if context:
            context_str = f"\n\nDetected Entities (Reference): {json.dumps(context, indent=2)}"

        prompt = f"""
        You are an expert medical AI assistant. Your task is to parse the following clinical note into a structured JSON format.
        
        Extract the following fields accurately:
        - patient_demographics (age, sex, etc.)
        - chief_complaint
        - history_of_present_illness (summary)
        - vital_signs: Normalize to a list of objects if multiple readings exist (e.g. [{{ "type": "blood_pressure", "value": "120/80", "timestamp": "current" }}]).
        - diagnosis (list of conditions)
        - medications (list of current meds)
        - plan (action items)

        Output ONLY valid JSON.
        
        Clinical Note:
        {text}
        {context_str}
        """

        url = "https://api.mistral.ai/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.mistral_key}",
            "Content-Type": "application/json",
            "Accept": "application/json"
        }
        data = {
            "model": "mistral-large-latest",
            "messages": [{"role": "user", "content": prompt}],
            "response_format": {"type": "json_object"}
        }

        try:
            response = requests.post(url, headers=headers, json=data)
            response.raise_for_status()
            result = response.json()
            content = result['choices'][0]['message']['content']
            parsed_json = json.loads(content)
            
            # Post-process for UI hints
            parsed_json["ui_hints"] = self.analyze_ui_hints(parsed_json)
            
            return parsed_json
        except Exception as e:
            logger.error(f"Mistral API Error: {e}")
            if 'response' in locals() and hasattr(response, 'text'):
                logger.error(f"API Response: {response.text}")
            return {"error": str(e)}

    def process_note(self, text: str, use_local: bool = True, use_llm: bool = True) -> Dict:
        """
        Full pipeline: BERT Extraction -> Mistral Parsing -> Merged Result.
        """
        result = {
            "raw_text": text,
            "bert_entities": [],
            "structured_data": {}
        }
        
        # Step 1: Local BERT
        if use_local:
            if not self.ner_pipeline:
                self.load_local_models()
            result["bert_entities"] = self.extract_entities_bert(text)
            
        # Step 2: Mistral LLM
        if use_llm:
            result["structured_data"] = self.parse_with_mistral(text, context=result["bert_entities"])
            
        return result

## 2. Initialize Models
We instantiate the parser and load the models.

In [3]:
# Initialize Parser
# TIP: Replace the empty string below with your actual API key if not in env vars
MISTRAL_KEY = os.environ.get("MISTRAL_API_KEY") or ""
parser = ClinicalNoteParser(mistral_api_key=MISTRAL_KEY)

# Load Local BioBERT Models (Note: multiple models around 400MB each)
print("Loading local models...")
parser.load_local_models(device=-1) # Set device=0 for GPU if available

2026-01-28 02:44:06,669 - __main__ - INFO - Mistral API Key detected.
2026-01-28 02:44:06,675 - __main__ - INFO - Loading local NER model: d4data/biomedical-ner-all...
2026-01-28 02:44:06,682 - __main__ - INFO - CUDA detected. Using GPU: NVIDIA GeForce RTX 3050 Laptop GPU


Loading local models...


2026-01-28 02:44:07,020 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/d4data/biomedical-ner-all/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-01-28 02:44:07,051 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/d4data/biomedical-ner-all/015a4050c9ac99722e61c547aa9b4282bcbedc7f/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

2026-01-28 02:44:07,646 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/d4data/biomedical-ner-all/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-01-28 02:44:08,174 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/d4data/biomedical-ner-all/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-01-28 02:44:08,806 - __main__ - INFO - Loading local Embedding model: emilyalsentzer/Bio_ClinicalBERT...
2026-01-28 02:44:09,060 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-01-28 02:44:09,085 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/emilyalsentzer/Bio_ClinicalBERT/d5892b39a4adaed74b92212a44081509db72f87b/config.json "HTTP/1.1 200 OK"
2026-01-28 02:44:09,641 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT/resolv

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-01-28 02:44:11,211 - httpx - INFO - HTTP Request: GET https://huggingface.co/api/models/emilyalsentzer/Bio_ClinicalBERT/commits/refs%2Fpr%2F16 "HTTP/1.1 200 OK"
BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-01-28 02:44:11,514 - httpx -

## 3. Semantic Search Demo
We can search for relevant patient history chunks based on meaning, not just exact keywords.

In [4]:
# Mock Corpus of Patient Histories / Reports
patient_histories = [
    "Pt with history of myocardial infarction (2019) and high cholesterol.",
    "Pt fell from ladder, sustaining localized trauma to left tibia.",
    "Severe allergic reaction to peanuts, required EpiPen.",
    "Chronic Type 2 Diabetes Mellitus managed with Metformin.",
    "Complains of irregular heartbeat and palpitations."
]

query = "heart problems"
print(f"--- Searching for: '{query}' ---")

results = parser.search_similar(query, patient_histories, top_k=3)
for r in results:
    print(f"Score {r['score']:.3f}: {r['text']}")

--- Searching for: 'heart problems' ---
Score 0.771: Pt with history of myocardial infarction (2019) and high cholesterol.
Score 0.763: Chronic Type 2 Diabetes Mellitus managed with Metformin.
Score 0.703: Severe allergic reaction to peanuts, required EpiPen.


## 4. Parsing with UI Triggers
Analyze a detailed note. The parser will output `ui_hints` to tell the Frontend which charts to display.

In [5]:
# Example Complex Note with Historical Vitals
complex_note = """
Pt: 55M. 
CC: Follow up for Hypertension.
HPI: Patient tracks BP at home. Readings last week: 145/90, 142/88. Today in clinic: 138/85.
Assessment: HTN slightly elevated but better than last month.
Plan: Continue Lisinopril. 
Alert: Monitor for dizziness.
"""

print("--- Processing Note ---")
result = parser.process_note(complex_note)

print("--- Structured Output (JSON) ---")
print(json.dumps(result["structured_data"], indent=2))

print("\n--- UI Logic Triggers ---")
print(json.dumps(result["structured_data"].get("ui_hints"), indent=2))

--- Processing Note ---
--- Structured Output (JSON) ---
{
  "patient_demographics": {
    "age": 55,
    "sex": "Male"
  },
  "chief_complaint": "Follow up for Hypertension",
  "history_of_present_illness": "Patient tracks blood pressure at home. Readings last week were 145/90 and 142/88. Today in clinic, blood pressure was 138/85.",
  "vital_signs": [
    {
      "type": "blood_pressure",
      "value": "145/90",
      "timestamp": "last_week"
    },
    {
      "type": "blood_pressure",
      "value": "142/88",
      "timestamp": "last_week"
    },
    {
      "type": "blood_pressure",
      "value": "138/85",
      "timestamp": "current"
    }
  ],
  "diagnosis": [
    "Hypertension (HTN)"
  ],
  "medications": [
    {
      "name": "Lisinopril",
      "status": "current"
    }
  ],
  "plan": [
    {
      "action": "Continue Lisinopril"
    },
    {
      "action": "Monitor for dizziness"
    }
  ],
  "ui_hints": {
    "show_bp_trend": true,
    "show_medication_list": true,
    "